# 07 — Search Subgraph

**Phase 7** — runs after `06_kg_construction.ipynb`.

## What this notebook does

1. **Pre-flight** — verifies Neo4j + Silra connectivity, vector index state, embedded chunk count.
2. **Embed probe** — embeds a single query, inspects vector shape and cosine norm.
3. **Vector search smoke** — runs 5 demo queries, displays top-5 hits each (chunk text + spine breadcrumb).
4. **Spine inspection** — for the top result, shows the full `TOPIC → DOCUMENT → CHAPTER → SECTION → PAGE → CHUNK` path.
5. **Keyword-graph expansion** — illustrates hybrid mode (vector + KW graph), shows delta vs pure vector.
6. **Tier & language filters** — demonstrates `tier_filter='primary'` and `language_filter='zh-classical'`.
7. **Result statistics** — tier distribution, language distribution, score histogram.
8. **Artefact write** — saves `notebooks/_artifacts/07_search/search.json`.
9. **Production note** — points to `scripts/run_search_demo.py`.

## Graph spine queried

```
(:CHUNK) <-[:HAS]- (:PAGE) <-[:INCLUDE]- (:SECTION)
         <-[:INCLUDE]- (:CHAPTER) <-[:CONSIST_OF]- (:DOCUMENT)
         <-[:CONTAIN]- (:TOPIC)
```

**Next**: Phase 7b — `07b_verifier.ipynb` (LLM evidence scoring).


In [ ]:
import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

# --- repo root on sys.path
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "apps").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    force=True,
)
logging.getLogger("neo4j.notifications").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "_artifacts" / "07_search"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"artifact  : {ARTIFACT_DIR}")

In [ ]:
from apps.backend.graph.neo4j_client import get_driver
from apps.backend.llm.silra import get_silra_client
from apps.backend.pipeline.search import (
    embed_query,
    vector_search,
    enrich_with_spine,
    keyword_expand,
    search,
    _extract_cjk_terms,
)

driver = get_driver()
silra_client = get_silra_client()
print("Neo4j driver ready")

## 1. Pre-flight checks

In [ ]:
preflight: dict = {}

with driver.session() as s:
    # Neo4j connectivity
    r = s.run("RETURN 1 AS ok").single()
    preflight["neo4j_ok"] = bool(r and r["ok"] == 1)

    # CHUNK counts
    r2 = s.run(
        "MATCH (c:CHUNK) "
        "RETURN count(c) AS total, "
        "       sum(CASE WHEN c.embeddingStatus='ok' THEN 1 ELSE 0 END) AS embedded"
    ).single()
    preflight["chunk_total"] = r2["total"]
    preflight["chunk_embedded"] = r2["embedded"]

    # Vector index state
    idx_rows = s.run(
        "SHOW INDEXES YIELD name, type, state "
        "WHERE name = 'chunk_embedding_vector_index'"
    ).data()
    preflight["vector_index"] = idx_rows[0] if idx_rows else {"state": "MISSING"}

    # KEYWORD count
    r3 = s.run("MATCH (k:KEYWORD) RETURN count(k) AS n").single()
    preflight["keyword_count"] = r3["n"]

    # MENTION edge count
    r4 = s.run("MATCH ()-[m:MENTION]->() RETURN count(m) AS n").single()
    preflight["mention_count"] = r4["n"]

print(json.dumps(preflight, indent=2))

assert preflight["neo4j_ok"], "Neo4j not reachable"
assert preflight["chunk_embedded"] > 0, "No embedded chunks — run 05_chunking_embeddings_bakeoff first"
assert preflight["vector_index"].get("state") == "ONLINE", (
    f"chunk_embedding_vector_index not ONLINE: {preflight['vector_index']}"
)
print("\n✅ Pre-flight passed")

## 2. Embed probe — inspect query vector shape

In [ ]:
import math

probe_query = "唐律中關於謀反罪的規定"
vec = embed_query(probe_query)

norm = math.sqrt(sum(x * x for x in vec))
print(f"Query      : {probe_query}")
print(f"Dims       : {len(vec)}")
print(f"L2 norm    : {norm:.6f}")
print(f"First 5    : {[round(x, 4) for x in vec[:5]]}")
print(f"Min / Max  : {min(vec):.4f} / {max(vec):.4f}")

## 3. Vector search smoke — 5 demo queries

In [ ]:
DEMO_QUERIES = [
    "唐律中謀反罪的刑罰規定",
    "唐代科舉制度與官員選拔",
    "均田制土地分配原則",
    "安史之亂後的財政改革",
    "唐代節度使的職權範圍",
]

search_records: list[dict] = []

for q in DEMO_QUERIES:
    results = search(driver, q, top_k=5, expand_keywords=True)
    search_records.append(
        {
            "query": q,
            "results": [r.to_dict() for r in results],
        }
    )
    print(f"\n{'─'*60}")
    print(f"Query: {q}  ({len(results)} results)")
    for r in results:
        tier = r.spine.document_tier or "?"
        doc  = (r.spine.document_title or "?")[:20]
        ch   = (r.spine.chapter_title or "")[:15]
        lang = r.hit.language or "?"
        print(
            f"  [{r.rank}] {r.score:.4f} | {tier:10s} | {lang:12s} | "
            f"{doc}{'·'+ch if ch else ''}\n"
            f"       {r.text[:120].replace(chr(10),' ')!r}"
        )

print(f"\n✅ {len(DEMO_QUERIES)} queries completed")

## 4. Full spine inspection — top result

In [ ]:
# Pick the top result from the first demo query
top_results = search(driver, DEMO_QUERIES[0], top_k=1, expand_keywords=False)
if top_results:
    r = top_results[0]
    s = r.spine
    print("Chunk ID   :", r.hit.chunk_id)
    print("Score      :", r.score)
    print("Citation   :", s.citation_label())
    print("Trust score:", s.trust_score())
    print()
    print("── TOPIC ─────────────────────────────")
    print(" topic_name          :", s.topic_name)
    print()
    print("── DOCUMENT ───────────────────────────")
    print(" document_title      :", s.document_title)
    print(" document_author     :", s.document_author)
    print(" document_tier       :", s.document_tier)
    print(" document_edition    :", s.document_edition)
    print(" document_editorial_layers:", s.document_editorial_layers)
    print(" publication_period  :", s.document_publication_period)
    print()
    print("── CHAPTER ────────────────────────────")
    print(" chapter_title       :", s.chapter_title)
    print(" chapter_ordinal     :", s.chapter_ordinal)
    print()
    print("── SECTION ────────────────────────────")
    print(" section_title       :", s.section_title)
    print(" section_ordinal     :", s.section_ordinal)
    print()
    print("── PAGE ───────────────────────────────")
    print(" page_id             :", s.page_id)
    print(" page_index          :", s.page_index)
    print(" page_language       :", s.page_language)
    print(" page_mode           :", s.page_mode)
    print()
    print("── CHUNK ──────────────────────────────")
    print(" chunk_id            :", r.hit.chunk_id)
    print(" char_count          :", r.hit.char_count)
    print(" language            :", r.hit.language)
    print(" keywords            :", s.keywords[:10])
    print()
    print("── TEXT ───────────────────────────────")
    print(r.text[:500])
else:
    print("No results — check embedding coverage")

## 5. Keyword-graph expansion — hybrid vs pure vector

In [ ]:
q = "唐代節度使的職權範圍"

pure_vector   = search(driver, q, top_k=10, expand_keywords=False)
hybrid        = search(driver, q, top_k=10, expand_keywords=True)

pure_ids  = {r.hit.chunk_id for r in pure_vector}
hybrid_ids = {r.hit.chunk_id for r in hybrid}
extra_ids  = hybrid_ids - pure_ids

print(f"Query         : {q}")
print(f"Pure vector   : {len(pure_vector)} results")
print(f"Hybrid        : {len(hybrid)} results")
print(f"New via KW    : {len(extra_ids)} chunks added by keyword graph expansion")

if extra_ids:
    print("\nKeyword-expansion additions:")
    for r in hybrid:
        if r.hit.chunk_id in extra_ids:
            print(
                f"  [KW] {r.spine.document_title or '?'} · "
                f"{r.spine.chapter_title or ''}: "
                f"{r.text[:100].replace(chr(10),' ')!r}"
            )

terms = _extract_cjk_terms(q)
print(f"\nCJK terms extracted from query: {terms[:15]}")

## 6. Tier and language filters

In [ ]:
q = "均田制土地分配原則"

primary_results = search(driver, q, top_k=5, tier_filter="primary")
secondary_results = search(driver, q, top_k=5, tier_filter="secondary")
classical_results = search(driver, q, top_k=5, language_filter="zh-classical")

print(f"Primary tier results   : {len(primary_results)}")
for r in primary_results:
    print(f"  {r.score:.4f} | {r.spine.document_title or '?'}")

print(f"\nSecondary tier results : {len(secondary_results)}")
for r in secondary_results:
    print(f"  {r.score:.4f} | {r.spine.document_title or '?'}")

print(f"\nzh-classical filter    : {len(classical_results)}")
for r in classical_results:
    print(f"  {r.score:.4f} | lang={r.hit.language} | {r.spine.document_title or '?'}")

## 7. Result statistics

In [ ]:
from collections import Counter

all_results = []
for rec in search_records:
    all_results.extend(rec["results"])

tier_dist = Counter(r.get("document_tier") for r in all_results)
lang_dist = Counter(r.get("language") for r in all_results)
scores    = [r["score"] for r in all_results if r.get("score")]

print("Tier distribution:")
for tier, cnt in tier_dist.most_common():
    print(f"  {tier:15s}: {cnt}")

print("\nLanguage distribution:")
for lang, cnt in lang_dist.most_common():
    print(f"  {lang:20s}: {cnt}")

if scores:
    print(f"\nScore range : {min(scores):.4f} – {max(scores):.4f}")
    print(f"Score mean  : {sum(scores)/len(scores):.4f}")

stats = {
    "tier_distribution": dict(tier_dist),
    "language_distribution": dict(lang_dist),
    "score_min": round(min(scores), 6) if scores else None,
    "score_max": round(max(scores), 6) if scores else None,
    "score_mean": round(sum(scores) / len(scores), 6) if scores else None,
}

## 8. Artefact write

In [ ]:
artifact = {
    "phase": "07_search_subgraph",
    "ts": datetime.now(timezone.utc).isoformat(),
    "preflight": preflight,
    "demo_queries": search_records,
    "statistics": stats,
}

artifact_path = ARTIFACT_DIR / "search.json"
artifact_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f"Artifact written → {artifact_path}")
print(f"File size: {artifact_path.stat().st_size:,} bytes")

## 9. Production runner note

The search pipeline is a **library** — it does not have a standalone long-running script in the same way as OCR or chunking.  Instead:

- Interactive queries: call `search(driver, query, ...)` directly in this notebook or the verifier notebook.
- Batch evaluation: see `10_bench_eval.ipynb` which runs a golden-set query suite.
- REST API (Phase 8): `apps/api/routes/search.py` will wrap `search()` in a FastAPI endpoint.

Embedding coverage as of this run: **{chunk_embedded} / {chunk_total}** chunks embedded.

To embed remaining chunks:

```bash
caffeinate -dimsu uv run python scripts/run_embedding.py \
    --batch-size 10 --log-file logs/run_embedding.log
```

**Next**: `07b_verifier.ipynb` — LLM evidence scoring over search results.
